In [1]:
import duckdb

# Connect to DuckDB in-memory database
con = duckdb.connect(database=':memory:')

print("=" * 70)
print("2. EXECUTING DATA CLEANING PIPELINE (FILTERING OUT MICRO-TRIPS & ANOMALIES)")
print("=" * 70)

cleaning_query = """
COPY (
    WITH base AS (
        SELECT 
            *,
            EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 60.0 AS duration_min,
            CASE 
                WHEN EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) > 0 
                THEN trip_distance / (EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime)) / 3600.0)
                ELSE NULL 
            END AS speed_mph
        FROM '../data/raw/yellow_tripdata_2025-*.parquet'
    )
    SELECT 
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        RatecodeID,
        store_and_fwd_flag,
        PULocationID,
        DOLocationID,
        payment_type,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        congestion_surcharge,
        Airport_fee,
        cbd_congestion_fee
    FROM base
    WHERE 
        -- 1. Valid Duration: strictly > 0 and <= 240 minutes (4 hours)
        duration_min > 0 
        AND duration_min <= 240 
        AND duration_min IS NOT NULL
        
        -- 2. Exclude Micro-Trips: duration must be at least 1.0 minute for non-zero distance trips
        AND NOT (duration_min < 1.0 AND duration_min > 0 and speed_mph > 70)
        
        -- 3. Valid Conditional Distance: <= 100 mi for intra-city, <= 200 mi for out-of-city (DO = 265)
        AND trip_distance > 0 
        AND (
            (DOLocationID != 265 AND trip_distance <= 100) 
            OR 
            (DOLocationID = 265 AND trip_distance <= 200)
        )
        
        -- 4. Valid Fare: non-negative
        AND fare_amount >= 0

        -- 5. Valid Location: Inside NYC Pickup Location And Known Locations
        AND PULocationID != 264 
        AND PULocationID != 265
        AND DOLocationID != 264

        -- 6. Valid Speed: strictly <= 80 mph (excludes GPS telemetry/jump errors)
        AND speed_mph IS NOT NULL
        AND speed_mph <= 80
        
        -- NOTE: Passenger count anomalies are retained per project policy.
) TO '../data/processed/yellow_tripdata_2025_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD);
"""

print("Applying cleaning filters and exporting clean Parquet file...")
con.execute(cleaning_query)
print("Successfully generated '../data/processed/yellow_tripdata_2025_clean.parquet'!")

# Verification query
verification_query = """
WITH raw_counts AS (
    SELECT COUNT(*) AS total_raw_rows FROM '../data/raw/yellow_tripdata_2025-*.parquet'
),
clean_counts AS (
    SELECT COUNT(*) AS total_clean_rows FROM '../data/processed/yellow_tripdata_2025_clean.parquet'
)
SELECT 
    total_raw_rows,
    total_clean_rows,
    total_raw_rows - total_clean_rows AS total_removed_rows,
    ROUND((total_raw_rows - total_clean_rows) * 100.0 / total_raw_rows, 2) AS removal_percentage
FROM raw_counts, clean_counts;
"""

df_summary = con.execute(verification_query).df()
con.close()

print("\n--- Final Cleaning Summary ---")
print(df_summary.to_string(index=False))

2. EXECUTING DATA CLEANING PIPELINE (FILTERING OUT MICRO-TRIPS & ANOMALIES)
Applying cleaning filters and exporting clean Parquet file...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Successfully generated '../data/processed/yellow_tripdata_2025_clean.parquet'!


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Final Cleaning Summary ---
 total_raw_rows  total_clean_rows  total_removed_rows  removal_percentage
       48722602          44015208             4707394                9.66
